# Random Forest na Classificação do Desempenho dos Alunos

# Seção 1: Importação do Dataset e Dataset Usado

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.ensemble import RandomForestClassifier

In [2]:
df = pd.read_parquet("Dados Limpos")

# Seção 4: Treinando o Random Forest

In [12]:
def treinar_rf(x_train, y_train, x_test, y_test, n_estimators, max_depth, max_features, min_samples_split, min_samples_leaf):
    
    rf= RandomForestClassifier(
        n_estimators=n_estimators,        
        max_depth=max_depth,            
        max_features = max_features,     
        min_samples_split = min_samples_split,    
        min_samples_leaf = min_samples_leaf,      
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    )

    rf.fit(x_train, y_train)

    y_pred_train = rf.predict(x_train)
    y_pred_test  = rf.predict(x_test)

    ein  = 1 - accuracy_score(y_train, y_pred_train)
    eout = 1 - accuracy_score(y_test,  y_pred_test)

    print(f"\nEin:  {ein:.4f}")
    print(f"Eout: {eout:.4f}")
    print(f"Gap:  {eout - ein:.4f}  {'overfitting' if eout - ein > 0.05 else 'ok'}")
    print("\n" + classification_report(y_test, y_pred_test))

    return rf

In [ ]:
df_reduzido = df.sample(n=100_000, random_state=42)

In [9]:
X = df_reduzido.drop(['NU_NOTA_CN', 'NU_NOTA_CH', 'NU_NOTA_LC','TP_SEXO', 'TP_COR_RACA',
       'NU_NOTA_MT', 'TX_RESPOSTAS_CN', 'TX_RESPOSTAS_CH', 'TX_RESPOSTAS_LC',
       'TX_RESPOSTAS_MT', 'TX_GABARITO_CN', 'TX_GABARITO_CH', 'TX_GABARITO_LC',
        'CLASSE_NOTA', 'MEDIA', 'TP_PRESENCA_CN', 'TP_PRESENCA_CH', 'TP_PRESENCA_LC', 'TP_PRESENCA_MT',
       'TX_GABARITO_MT', 'TP_STATUS_REDACAO', 'NU_NOTA_COMP1', 'NU_NOTA_COMP2','IN_TREINEIRO', 
       'NU_NOTA_COMP3', 'NU_NOTA_COMP4', 'NU_NOTA_COMP5', 'NU_NOTA_REDACAO'], axis=1)

y = df_reduzido['CLASSE_NOTA']

In [11]:
x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [24]:
rf = RandomForestClassifier(random_state=42, class_weight='balanced')

rf.fit(x_train, y_train)

print('Ein: %0.4f' % (1 - accuracy_score(y_train, rf.predict(x_train))))
print('Eout: %0.4f' % (1 - accuracy_score(y_test, rf.predict(x_test))))
print(classification_report(y_test, rf.predict(x_test)))

Ein: 0.0013
Eout: 0.2738
              precision    recall  f1-score   support

           0       0.73      0.69      0.71      9725
           1       0.72      0.76      0.74     10275

    accuracy                           0.73     20000
   macro avg       0.73      0.73      0.73     20000
weighted avg       0.73      0.73      0.73     20000



## Tratando Overfit

In [13]:
n_estimators=60
max_depth=30          
max_features='log2'    
min_samples_split=50
min_samples_leaf=10
    
rf = treinar_rf(x_train, y_train, x_test, y_test, n_estimators, max_depth, max_features, min_samples_split, min_samples_leaf)



Ein:  0.2385
Eout: 0.2708
Gap:  0.0323  ok

              precision    recall  f1-score   support

           0       0.76      0.68      0.72     10030
           1       0.71      0.78      0.74      9970

    accuracy                           0.73     20000
   macro avg       0.73      0.73      0.73     20000
weighted avg       0.73      0.73      0.73     20000



In [14]:
imp = pd.Series(rf.feature_importances_, index=X.columns)
imp = imp.sort_values(ascending=False)

print(imp.head(20))
print(imp.head(20).sum())

TP_DEPENDENCIA_ADM_ESC    0.129780
INDICE_SOCIO              0.116878
TIPO_ESCOLA_COMP          0.102472
Q006                      0.069747
TP_ESCOLA                 0.067421
Q024                      0.063009
CAPITAL_EDUCACIONAL       0.062273
Q010                      0.051987
RECURSOS_ESTUDO           0.030671
Q002                      0.027792
ESCOLARIDADE_PAIS         0.024432
TP_FAIXA_ETARIA           0.024198
NU_ANO                    0.023298
Q003                      0.023101
Q001                      0.019630
Q004                      0.017942
Q008                      0.017545
Q005                      0.016385
Q013                      0.014600
Q018                      0.013461
dtype: float64
0.9166216907530971
